In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# architecture of encoder
encoder = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(8, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(8, 8, kernel_size=3),
    nn.ReLU(),
    nn.Conv2d(8, 8, kernel_size=3),
    nn.Sigmoid(),
    nn.Flatten()
)

In [ ]:
class Classifier(nn.Module):
    def __init__(self, encoder, num_features, num_classes, num_hidden=512):
        super().__init__()
        self.encoder = encoder
        self.perceptron = nn.Sequential(
            nn.Linear(num_features,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_classes),
        )
    def forward(self, x):
        features = self.encoder(x)
        logits = self.perceptron(features)
        if self.training:
            return logits
        else:
            return F.softmax(logits, dim=1)

In [ ]:
classifier = Classifier(encoder,72,10).to(device)

In [ ]:
!wget http://agentspace.org/download/mnist_classifier.pth
model_name = 'mnist_classifier.pth'
classifier.load_state_dict(torch.load(model_name, map_location=device))

In [ ]:
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())

In [ ]:
sample = test_dataset[0][0]
print(sample.shape)
plt.imshow(sample.squeeze(0), cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
with torch.no_grad():
    logits = classifier(sample.unsqueeze(0).to(device)).squeeze(0).detach().cpu().numpy()

print(logits.shape)
print(logits)
category = np.argmax(logits)
print(category)

In [ ]:
def softmax(x):
    # subtract max for numerical stability
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

print(np.round(softmax(logits),3))

In [ ]:
scale = 4
print(scale)
print(np.round(softmax(logits/scale),3))

In [ ]:
class ScaledSoftmax(nn.Module):
    def __init__(self, dim=1, scale=1.0):
        super().__init__()
        self.scale = scale
        self.softmax = nn.Softmax(dim=dim)

    def forward(self, x):
        return self.softmax(x / self.scale)

In [ ]:
classifier

In [ ]:
# convert the perceptron into convolutional layers
fc1 = classifier.perceptron[0]  # Linear(72 → 512)
fc2 = classifier.perceptron[2]  # Linear(512 → 10)
conv_fc1 = nn.Conv2d(8, fc1.out_features, kernel_size=3, bias=True).to(device)
conv_fc2 = nn.Conv2d(fc1.out_features, fc2.out_features, kernel_size=1, bias=True).to(device)
# copy weights
with torch.no_grad():
    conv_fc1.weight.copy_(fc1.weight.view(fc1.out_features, 8, 3, 3))
    conv_fc1.bias.copy_(fc1.bias)
    conv_fc2.weight.copy_(fc2.weight.view(fc2.out_features, fc2.in_features, 1, 1))
    conv_fc2.bias.copy_(fc2.bias)
# create fully-convolutional perceptron
fully_convolutional_perceptron = nn.Sequential(
    conv_fc1,
    nn.ReLU(),
    conv_fc2
)
# remove flattening from the encoder
encoder_without_flattening = nn.Sequential(*list(classifier.encoder.children())[:-1])
# compose the new model
model = nn.Sequential(
    encoder_without_flattening,
    fully_convolutional_perceptron,
    ScaledSoftmax(dim=1,scale=4)
)
model

In [ ]:
with torch.no_grad():
    probabilities = model(sample.unsqueeze(0).to(device)).squeeze(0).detach().cpu().numpy()

print(probabilities.shape)
print(np.round(probabilities[:,0,0],3))
category = np.argmax(probabilities[:,0,0])
confidence = np.max(probabilities[:,0,0])
print(category,confidence)

In [ ]:
# Save model and weights
def save():
    torch.save(model.state_dict(), model_name) # weights only
    print(f'Saved trained model at {model_name}')

In [ ]:
model_name = 'mnist_simple_detector.pth'
save()

In [ ]:
# download model
from google.colab import files
files.download(model_name)

In [ ]:
def shift(sample,s):
    shifted = torch.zeros_like(sample, device=sample.device)
    shifted[...,s[0]:,s[1]:] = sample[...,:-s[0],:-s[1]]
    return shifted

In [ ]:
shifted = torch.stack([shift(sample,(i,i)) for i in range(1,16)])
plt.figure(figsize=(20, 4))
for i, img in enumerate(shifted):
    plt.subplot(1, len(shifted), i+1)
    plt.imshow(img.squeeze(0).detach().cpu().numpy(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    probabilities = model(shifted.to(device)).detach().cpu().numpy()

category = np.argmax(probabilities[:,:,0,0],axis=1)
confidence = np.max(probabilities[:,:,0,0],axis=1)
print(category)
print(confidence)

In [ ]:
sz = 224  # == 8 * 28
patch_sz = 28
image = torch.zeros(1,sz,sz,device=device)
for _ in range(12):
    i = np.random.randint(0, len(train_dataset))
    x, y = np.random.randint(0, sz-patch_sz), np.random.randint(0, sz-patch_sz)
    if (image[:,y:y+patch_sz, x:x+patch_sz]>0.5).any():
        continue
    image[:,y:y+patch_sz, x:x+patch_sz] += train_dataset[i][0].to(image.device)

In [ ]:
plt.imshow(image.squeeze(0).detach().cpu().numpy(), cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
out = model(image.unsqueeze(0))
print(out.shape)

In [ ]:
confidencemap = out.max(dim=1).values
print(confidencemap.shape)

In [ ]:
heatmap = confidencemap.float().squeeze(0).detach().cpu().numpy()
plt.imshow(heatmap, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
import torch
import torch.nn.functional as F

def NonMaximumSuppression(heatmap, radius):
    heatmap = heatmap.clone()
    H, W = heatmap.shape[1:]

    suppressed = torch.zeros_like(heatmap)
    results = []

    while True:
        # Find max value and position
        max_val = heatmap.max()
        if max_val <= 0:
            break

        max_pos = torch.nonzero(heatmap == max_val)[0]
        _, y, x = max_pos
        results.append((y.item(), x.item(), max_val.item()))

        # Suppress region around the peak
        y0 = max(0, y - radius)
        y1 = min(H, y + radius + 1)
        x0 = max(0, x - radius)
        x1 = min(W, x + radius + 1)

        heatmap[:, y0:y1, x0:x1] = 0

    return results

In [ ]:
detections = NonMaximumSuppression(confidencemap, 6)
print(detections)


In [ ]:
disp = cv2.cvtColor(heatmap,cv2.COLOR_GRAY2BGR)
for y, x, confidence in detections:
    cv2.circle(disp,(x,y),1,(0,0,255),cv2.FILLED)

In [ ]:
plt.imshow(disp)
plt.axis('off')
plt.show()